# E24 · Pilot bukti konteks soal 10

Uji apakah model NLI multibahasa bisa membedakan kejadian di sekolah, di rumah,
dan lokasi yang tidak jelas. Jalankan dari atas. T4 opsional, CPU juga bisa.

Model frozen membaca pasangan **teks → pernyataan tentang lokasi**. Ia memberi
tiga skor NLI untuk setiap pernyataan: implikasi, netral, dan kontradiksi.
Untuk tiap kriteria, notebook memperlihatkan kutipan dengan skor implikasi
tertinggi. Kutipan itu kandidat bukti yang perlu dibaca, bukan alasan otomatis.

Pilot memakai development E21 dan workbook semi-final yang sama. Skor guru
digunakan setelah inferensi untuk mengatur tabel audit, tidak masuk ke model.
ZIP audit mengandung kutipan jawaban. Simpan di Drive privat.

[Rujukan tugas per kriteria · TRATES](https://aclanthology.org/2025.findings-acl.1054/)
menjadi inspirasi eksploratif. Bukti kuartil venue belum terverifikasi.
[García-Varela dkk.](https://doi.org/10.3390/educsci15080946) memberi konteks
metodologi kriteria eksplisit. NLI E24 adalah adaptasi lokal. Pilot ini berhenti
setelah audit bukti, sebelum keputusan validasi dan training S3.


## 1 · Setup runtime

In [ ]:
# Jalankan pada runtime Colab baru, sebelum mengimpor pustaka numerik/model.
import sys
import subprocess
from importlib import metadata, util
from packaging.requirements import Requirement

# Pertahankan Torch/CUDA, NumPy, pandas, SciPy dan sklearn bawaan Colab.
# Mengganti pustaka numerik yang sudah dimuat kernel dapat menyebabkan konflik ABI.
CORE_REQUIREMENTS = {
    'torch': 'torch>=2.6', 'numpy': 'numpy>=1.26,<3',
    'pandas': 'pandas>=2,<3', 'scipy': 'scipy>=1.10,<2',
    'scikit-learn': 'scikit-learn>=1.2,<2',
}
for package, requirement in CORE_REQUIREMENTS.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError as error:
        raise RuntimeError('Gunakan runtime Google Colab baru; pustaka dasar belum ada: '+package) from error
    if not Requirement(requirement).specifier.contains(installed):
        raise RuntimeError(requirement+' diperlukan; runtime ini memiliki '+installed+'. Pilih runtime Colab baru.')

# Tokenizers 0.19 tidak menyediakan wheel Python 3.13; tetap satu versi per runtime.
TRANSFORMERS_REQUIREMENT = 'transformers==4.48.3' if sys.version_info >= (3, 13) else 'transformers==4.44.2'
EXTRA_REQUIREMENTS = [TRANSFORMERS_REQUIREMENT, 'openpyxl>=3.1,<4', 'sentencepiece>=0.2,<1', 'protobuf>=4,<7']
to_install = []
for requirement in EXTRA_REQUIREMENTS:
    req = Requirement(requirement)
    try:
        installed = metadata.version(req.name)
    except metadata.PackageNotFoundError:
        installed = None
    if installed is None or not req.specifier.contains(installed):
        if req.name in sys.modules:
            raise RuntimeError(req.name+' sudah diimpor dengan versi berbeda. Restart session, lalu Run all dari atas.')
        to_install.append(requirement)
if to_install:
    # Kendala ini mencegah pip mengganti pustaka numerik yang digunakan kernel.
    from pathlib import Path
    constraints = Path('/tmp/aes35_native_constraints.txt')
    constraints.write_text(''.join(p+'=='+metadata.version(p)+'\n' for p in CORE_REQUIREMENTS))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           '--constraint', str(constraints), *to_install])
print('Setup selesai. Semua cell berikutnya berjalan langsung di kernel notebook ini.')
print('Python:', sys.version.split()[0])


In [ ]:
import os, sys, json, math, hashlib, random, warnings, re, itertools, unicodedata, zipfile, subprocess, gc, logging, html
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.functional import normalize
from torch.utils.data import DataLoader
def configure_text_only_transformers():
    """Hindari import vision ops yang tidak diperlukan oleh RoBERTa/BGE/LoRA."""
    import os
    os.environ['USE_TORCH'] = '1'
    os.environ['USE_TF'] = '0'
    os.environ['USE_FLAX'] = '0'
    from transformers.utils import import_utils
    # Transformers 4.44/4.48 mendeteksi torchvision dari keberadaan paket,
    # meskipun binary ops-nya tidak cocok dengan torch runtime. Notebook ini
    # hanya teks: jangan muat backend gambar melalui import tidak langsung PEFT.
    # Backend teks dikonfigurasi pada kernel notebook; tidak memuat operator vision.
    if not hasattr(import_utils, '_torchvision_available'):
        raise RuntimeError('Versi Transformers berbeda dari versi notebook yang didukung.')
    import_utils._torchvision_available = False
    return {'transformers_backend': 'pytorch_text_only', 'torchvision_enabled': False}

configure_text_only_transformers()
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from scipy.stats import pearsonr
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from openpyxl import Workbook, load_workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
SCENARIOS = ('S3',)  # Hanya model fusion yang dijalankan.
NOTEBOOK_CODE_SHA256='d9dd665f8bdb8e3726d60a9f618be723a45f2a3239c64ac6b5673d4b3dadf260'

from IPython.display import display, HTML


## 2 · Siapkan split development E21

In [ ]:
SCOPE = 'reset_e20_e19_checkpoint_ab'
SHEET_NAME = 'Sheet1'
QUESTION_COLUMN = 'Pertanyaan'
REFERENCE_COLUMN = 'Kunci_Jawaban'
ANSWER_COLUMN = 'Jawaban'
GROUP_COLUMN = 'Email Address'
RESPONSE_NUMBER_COLUMN = 'No_Soal'
INPUT_COLUMNS = (QUESTION_COLUMN, REFERENCE_COLUMN, ANSWER_COLUMN)

# Setel sesuai pedoman penilaian sebelum training. Rentang awal sesuai kategori di file contoh.
TRAITS = (
    ('isi', 'Skor_Isi', 1, 4),
    ('organisasi', 'Skor_Organisasi', 1, 2),
    ('kebahasaan', 'Skor_Tata_Bahasa', 1, 2),
    ('mekanik', 'Skor_Mekanik', 1, 2),
)
TARGET_COLUMNS = tuple(column for _, column, _, _ in TRAITS)
REQUIRED = (*INPUT_COLUMNS, GROUP_COLUMN, RESPONSE_NUMBER_COLUMN, *TARGET_COLUMNS)
RUBRIC = {'traits': [
    {'name': name, 'min': low, 'max': high, 'categories': list(range(low, high + 1))}
    for name, _, low, high in TRAITS
]}
SCORE_MIN = np.asarray([low for _, _, low, _ in TRAITS], dtype=np.float32)
SCORE_SPAN = np.asarray([high - low for _, _, low, high in TRAITS], dtype=np.float32)


In [ ]:
# Baca dan audit data tanpa mengubah file Excel asli.
# Rapikan Unicode dan karakter kontrol; ejaan, huruf besar, dan tanda baca jawaban tetap dipertahankan.
def clean(text, normalize_punctuation=False):
    text = unicodedata.normalize('NFC', text).replace('\r\n','\n').replace('\r','\n')
    text = ''.join(c for c in text if c in '\n\t' or unicodedata.category(c) != 'Cc')
    if normalize_punctuation:
        text = text.translate(str.maketrans({'“':'"','”':'"','‘':"'",'’':"'",'–':'-','—':'-'})).replace('…','...')
    return text

def digest(path):
    # Checkpoint besar diperiksa bertahap supaya tidak dimuat seluruhnya ke RAM.
    checksum = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8*1024*1024), b''):
            checksum.update(chunk)
    return checksum.hexdigest()


def dump(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+'.pending')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)+'\n', encoding='utf-8')
    temporary.replace(path)


@dataclass
class Bundle:
    rows: list
    y: np.ndarray
    groups: np.ndarray
    source_sha256: str
    summary: dict

    @property
    def texts(self):
        return [r['text'] for r in self.rows]


# Baca jawaban dan nilai guru, periksa kelengkapan, lalu kelompokkan email/duplikat untuk mencegah kebocoran data.
def read_dataset(cfg):
    path = Path(cfg['dataset'])
    source_hash = digest(path)
    book = load_workbook(path, read_only=True, data_only=True)
    try:
        if cfg['sheet'] not in book.sheetnames:
            raise ValueError(f"Tab {cfg['sheet']} tidak tersedia: {book.sheetnames}")
        data = iter(book[cfg['sheet']].iter_rows(values_only=True))
        header = [str(x).strip() if x is not None else None for x in next(data, ())]
        if any(header.count(x) != 1 for x in REQUIRED):
            raise ValueError('Header wajib hilang atau duplikat: '+', '.join(REQUIRED))
        rows = []
        for excel_row, values in enumerate(data, 2):
            if all(v is None for v in values):
                continue  # Hanya baris format kosong, bukan label/jawaban yang hilang.
            raw = {k: values[i] for i, k in enumerate(header) if k is not None}
            missing = [k for k in REQUIRED if raw[k] is None or not str(raw[k]).strip()]
            if missing:
                raise ValueError(f'Baris {excel_row}: kolom wajib kosong: '+', '.join(missing))
            scores = []
            for _, column, low, high in TRAITS:
                score = raw[column]
                if (isinstance(score, bool) or not isinstance(score, (int, float))
                    or not math.isfinite(score) or int(score) != score or not low <= score <= high):
                    raise ValueError(f'Baris {excel_row}: {column} harus bilangan bulat {low}–{high}.')
                scores.append(int(score))
            for key in (QUESTION_COLUMN,REFERENCE_COLUMN):
                if not isinstance(raw[key],str) or not raw[key].strip():
                    raise ValueError(f'Baris {excel_row}: {key} harus teks yang tidak kosong.')
            if not isinstance(raw[ANSWER_COLUMN], str) or not clean(raw[ANSWER_COLUMN]).strip():
                raise ValueError(f'Baris {excel_row}: jawaban bukan teks.')
            rows.append({'row_id': f'row_{excel_row-1:04d}', 'source_row': excel_row,
                         'raw_text': raw[ANSWER_COLUMN], 'text': clean(raw[ANSWER_COLUMN]),
                         'email_group': unicodedata.normalize('NFC', str(raw[GROUP_COLUMN])).strip().casefold(),
                         'question': clean(raw[QUESTION_COLUMN]), 'reference': clean(raw[REFERENCE_COLUMN]),
                         'response_number': str(raw[RESPONSE_NUMBER_COLUMN]), 'scores': scores})
    finally:
        book.close()
    if cfg.get('expected_rows') is not None and len(rows) != cfg['expected_rows']:
        raise ValueError(f"Diharapkan {cfg['expected_rows']} baris, ditemukan {len(rows)}; tidak dipotong otomatis.")
    if not rows:
        raise ValueError('Dataset kosong')
    parent = list(range(len(rows)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        parent[find(b)] = find(a)

    email_first, duplicate_sets = {}, {}
    for i, row in enumerate(rows):
        email = row['email_group']
        if email in email_first:
            union(email_first[email], i)
        email_first[email] = i
        # Normalisasi ini HANYA audit kesamaan, bukan teks masukan encoder.
        normalized = ' '.join(row['text'].casefold().split())
        key = hashlib.sha256(normalized.encode()).hexdigest()
        duplicate_sets.setdefault(key, []).append(i)
    for indices in duplicate_sets.values():
        for i in indices[1:]:
            union(indices[0], i)
    roots = {root: i for i, root in enumerate(sorted({find(i) for i in range(len(rows))}))}
    groups = np.asarray([roots[find(i)] for i in range(len(rows))], dtype=int)
    y = np.asarray([row.pop('scores') for row in rows], dtype=np.float32)
    summary = {
        'scope': SCOPE, 'rows': len(rows), 'target_columns': list(TARGET_COLUMNS),
        'model_outputs': len(TRAITS), 'categories': RUBRIC['traits'],
        'email_groups': len(email_first), 'connected_groups': len(roots),
        'questions': len({row['response_number'] for row in rows}),
        'duplicate_sets': sum(len(indices) > 1 for indices in duplicate_sets.values()),
        'source_sha256': source_hash, 'input_features': list(INPUT_COLUMNS),
    }
    for row in rows:
        del row['email_group']
    return Bundle(rows, y, groups, source_hash, summary)

# Identitas hasil mengikuti data, konfigurasi, dan kode notebook ini.
def run_folder(cfg, bundle):
    code_hash = NOTEBOOK_CODE_SHA256
    semantic = {k: v for k, v in cfg.items() if k not in ('dataset', 'output', 'trace_position', 'vector_source', 'new_answer', 'focus_word', 'new_question', 'new_reference')}
    key = hashlib.sha256(json.dumps({'source': bundle.source_sha256, 'config': semantic, 'code': code_hash}, sort_keys=True).encode()).hexdigest()[:16]
    folder = Path(cfg['output']) / key
    folder.mkdir(parents=True, exist_ok=True)
    dump(folder / 'effective_config.json', cfg)
    return folder


In [ ]:
def make_split(bundle, cfg):
    indices = np.arange(len(bundle.rows))
    splitter = GroupShuffleSplit(n_splits=cfg['split_candidates'], test_size=cfg['test_fraction'], random_state=cfg['split_seed'])
    # Pilih kedekatan JUMLAH BARIS dengan target 80:20, bukan skor/hasil model.
    # GroupShuffleSplit membagi kelompok; ukuran kelompok tidak selalu sama.
    candidates = list(splitter.split(indices, groups=bundle.groups))
    development, test = min(candidates, key=lambda pair: abs(len(pair[1])/len(indices)-cfg['test_fraction']))
    if len(set(bundle.groups[development])) < cfg['folds']:
        raise ValueError('Kelompok training tidak cukup untuk CV yang diminta.')
    folds = []
    for number, (a, b) in enumerate(GroupKFold(n_splits=cfg['folds']).split(development, groups=bundle.groups[development]), 1):
        training, validation = development[a], development[b]
        assert not set(bundle.groups[training]) & set(bundle.groups[validation])
        folds.append({'fold': number, 'train': training.tolist(), 'validation': validation.tolist()})
    assert not set(bundle.groups[development]) & set(bundle.groups[test])
    manifest = {'scope': SCOPE, 'source_sha256': bundle.source_sha256,
                'development': development.tolist(), 'test': test.tolist(), 'folds': folds,
                'development_rows': len(development), 'test_rows': len(test),
                'development_groups': len(set(bundle.groups[development])),
                'test_groups': len(set(bundle.groups[test])),
                'target_test_row_fraction': cfg['test_fraction'],
                'candidate_group_fraction': cfg['test_fraction'],
                'split_candidates': cfg['split_candidates'],
                'selection_uses_labels': False,
                'actual_test_row_fraction': len(test)/len(indices),
                'policy': 'Holdout kelompok: pilih kandidat deterministik terdekat target proporsi baris berdasarkan ukuran kelompok saja, tanpa label/hasil model. CV hanya di development. Email/duplikat tidak menyeberang. Mengukur kelompok baru pada pertanyaan ini, bukan unseen-prompt.',
                'test_used_for_selection': False}
    manifest['split_sha256'] = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    return manifest


In [ ]:
# Notebook eksperimen berikutnya bisa memakai path dan SHA-256 yang sama.
DATASET_VERSION = 'semi-final'
DATASET_NAME = 'data_cleaned_3sekolah.xlsx'
DATASET_SHA256 = 'dfbd97128b09a4eb488dd890ccc3efd9b9fe232e74fcc00ee5504a03acc58226'
DRIVE_MOUNT = Path('/content/drive')
DRIVE_DATASET = DRIVE_MOUNT / 'MyDrive' / 'AES' / 'datasets' / 'semi_final' / DATASET_NAME
LOCAL_DATASET = Path(DATASET_NAME)  # Untuk eksekusi di luar Colab.
OUTPUT_NAME = 'aes_reset_e24_pilot_nli'
IN_COLAB = 'google.colab' in sys.modules or Path('/content').is_dir()
OUTPUT_ROOT = (DRIVE_MOUNT / 'MyDrive' / 'AES' / 'outputs' / OUTPUT_NAME
               if IN_COLAB else Path('outputs') / OUTPUT_NAME)


def resolve_colab_dataset(mount_drive, upload, drive_mount, drive_dataset):
    """Gunakan salinan privat Drive atau simpan workbook yang cocok pada upload pertama."""
    from tempfile import NamedTemporaryFile

    mount_drive(str(drive_mount))
    if not (drive_mount / 'MyDrive').is_dir():
        raise RuntimeError('MyDrive belum tersedia. Hubungkan Google Drive di Colab.')
    if not drive_dataset.exists():
        print(f'Unggah {DATASET_NAME} untuk penyimpanan pertama di Drive.')
        uploads = upload()
        if len(uploads) != 1:
            raise ValueError(f'Unggah satu workbook asli bernama {DATASET_NAME}.')
        workbook = next(iter(uploads.values()))
        if hashlib.sha256(workbook).hexdigest() != DATASET_SHA256:
            raise ValueError('File unggahan berbeda dari workbook semi-final yang dipakai eksperimen ini.')
        drive_dataset.parent.mkdir(parents=True, exist_ok=True)
        with NamedTemporaryFile(dir=drive_dataset.parent, suffix='.pending', delete=False) as stream:
            staging = Path(stream.name)
            try:
                stream.write(workbook)
                stream.close()
                if digest(staging) != DATASET_SHA256:
                    raise IOError('Salinan di Drive gagal diverifikasi.')
                if not drive_dataset.exists():
                    staging.replace(drive_dataset)
            finally:
                staging.unlink(missing_ok=True)
        print(f'Workbook semi-final tersedia di Drive: {drive_dataset}')
    return drive_dataset


if 'google.colab' in sys.modules or Path('/content').is_dir():
    from google.colab import drive, files
    DATASET_PATH = resolve_colab_dataset(drive.mount, files.upload, DRIVE_MOUNT, DRIVE_DATASET)
else:
    DATASET_PATH = LOCAL_DATASET

if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'Workbook semi-final belum ditemukan: {DATASET_PATH}')
if digest(DATASET_PATH) != DATASET_SHA256:
    raise ValueError(f'Workbook di {DATASET_PATH} berbeda dari versi semi-final terpilih. '
                     'Gunakan file asli atau tetapkan path dan SHA-256 versi baru secara eksplisit.')
print(f'Dataset {DATASET_VERSION} terverifikasi: {DATASET_PATH}')


In [ ]:
cfg = {
    'dataset': str(DATASET_PATH), 'sheet': SHEET_NAME, 'expected_rows': 1340,
    'test_fraction': 0.20, 'folds': 5, 'split_seed': 2026, 'split_candidates': 128,
}
bundle = read_dataset(cfg)
manifest = make_split(bundle, cfg)
EXPECTED_SPLIT = 'f7fba02006ae61f96fa586fb6b87de678038b38679ac0629af101cb66e69c16c'
if manifest['split_sha256'] != EXPECTED_SPLIT:
    raise ValueError('Split pilot berbeda dari E21. Periksa dataset dan versi scikit-learn.')
indices = [i for i in manifest['development'] if bundle.rows[i]['response_number'].strip() == '10']
if len(indices) != 107:
    raise ValueError('Diharapkan 107 jawaban development untuk soal 10.')
print(f'{len(indices)} jawaban soal 10 dari {len(manifest["development"])} jawaban development.')


## 3 · Muat model NLI dan definisikan kriteria

In [ ]:
MODEL_ID = 'MoritzLaurer/mDeBERTa-v3-base-mnli-xnli'
MODEL_REVISION = 'main'  # SHA efektif diputuskan sekali dan ditulis ke artefak.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH = 8 if DEVICE == 'cuda' else 4
MAX_TOKENS = 384
CRITERIA = {
    'sekolah': 'Kejadian yang diceritakan berlangsung di sekolah.',
    'rumah': 'Kejadian yang diceritakan berlangsung di rumah.',
}


In [ ]:
def label_indices(config):
    names = {str(name).lower(): int(i) for i, name in config.id2label.items()}
    required = {'entailment', 'neutral', 'contradiction'}
    if set(names) != required or len(set(names.values())) != 3:
        raise ValueError(f'Label NLI tidak dikenali: {config.id2label}')
    return names


class EvidenceReader:
    def __init__(self, model, tokenizer, device, batch_size, max_tokens):
        self.model = model.eval().requires_grad_(False)
        self.tokenizer = tokenizer
        self.device = device
        self.batch_size = batch_size
        self.max_tokens = max_tokens
        self.labels = label_indices(model.config)
        # Sediakan ruang untuk hipotesis terpanjang dan tiga token pembatas.
        hypothesis_width = max(len(tokenizer(value, add_special_tokens=False)['input_ids'])
                               for value in CRITERIA.values())
        self.premise_cap = max_tokens - hypothesis_width - tokenizer.num_special_tokens_to_add(pair=True)
        if self.premise_cap <= 48:
            raise ValueError('Jendela premis terlalu sempit.')

    def passages(self, text):
        chunks = []
        for sentence in re.split(r'(?<=[.!?])\s+|\n+', text):
            sentence = sentence.strip()
            if not sentence:
                continue
            offsets = self.tokenizer(sentence, add_special_tokens=False, return_offsets_mapping=True)['offset_mapping']
            if len(offsets) <= self.premise_cap:
                chunks.append(sentence)
                continue
            for start in range(0, len(offsets), self.premise_cap - 32):
                window = offsets[start:start + self.premise_cap]
                chunks.append(sentence[window[0][0]:window[-1][1]])
                if start + self.premise_cap >= len(offsets):
                    break
        return chunks or [text]

    @torch.inference_mode()
    def probabilities(self, premises, hypotheses):
        scores = []
        for start in range(0, len(premises), self.batch_size):
            end = start + self.batch_size
            x = self.tokenizer(premises[start:end], hypotheses[start:end],
                               max_length=self.max_tokens, truncation='only_first',
                               padding=True, return_tensors='pt')
            logits = self.model(**x.to(self.device)).logits.float()
            scores.extend(logits.softmax(dim=-1).cpu().tolist())
        return scores

    def features(self, text):
        text = text.strip()
        if not text:
            raise ValueError('Jawaban kosong.')
        full_size = len(self.tokenizer(text, add_special_tokens=False)['input_ids'])
        full_available = full_size <= self.premise_cap
        windows = self.passages(text)
        views = [text, *windows] if full_available else windows
        pairs = [(view, hypothesis) for hypothesis in CRITERIA.values() for view in views]
        values = self.probabilities([pair[0] for pair in pairs], [pair[1] for pair in pairs])
        result = {'jawaban_utuh_dinilai': full_available, 'jumlah_kutipan': len(windows)}
        for i, key in enumerate(CRITERIA):
            block = values[i * len(views):(i + 1) * len(views)]
            for label, column in self.labels.items():
                result[f'{key}_utuh_{label}'] = block[0][column] if full_available else None
            snippet_values = block[int(full_available):]
            best = max(range(len(windows)), key=lambda index: snippet_values[index][self.labels['entailment']])
            result[f'{key}_kutipan_kandidat'] = windows[best]
            result[f'{key}_kutipan_entailment'] = snippet_values[best][self.labels['entailment']]
        return result


In [ ]:
from huggingface_hub import HfApi
from transformers import AutoModelForSequenceClassification

model_info = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION)
revision = model_info.sha
if not revision or not re.fullmatch(r'[0-9a-f]{40}', revision):
    raise ValueError('SHA revisi pretrained tidak ditemukan.')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision, use_fast=True)
model_files = {entry.rfilename for entry in model_info.siblings}
use_safetensors = any(name.endswith('.safetensors') for name in model_files)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, revision=revision, use_safetensors=use_safetensors,
).to(DEVICE)
if not tokenizer.is_fast:
    raise RuntimeError('Pilot memerlukan tokenizer fast untuk memetakan kutipan ke teks asli.')
reader = EvidenceReader(model, tokenizer, DEVICE, BATCH, MAX_TOKENS)
print(f'Model frozen: {MODEL_ID} @ {revision} · {DEVICE}')


## 4 · Cek contoh buatan

In [ ]:
# Contoh buatan untuk memeriksa arah fitur sebelum membuka kasus development.
probes = [
    ('Sekolah langsung', 'Para siswa kerja bakti membersihkan halaman sekolah.', 'sekolah'),
    ('Rumah langsung', 'Kami kerja bakti di rumah bersama keluarga.', 'rumah'),
    ('Sekolah tersirat', 'Seusai bel pelajaran, kami menyapu ruang kelas bersama wali kelas.', 'sekolah'),
    ('Rumah tersirat', 'Ayah dan ibu membersihkan ruang tamu bersama saya dan adik.', 'rumah'),
    ('Sekolah disebut sebagai pembanding', 'Di sekolah biasanya ada lomba. Kali ini kerja bakti kami berlangsung di rumah.', 'rumah'),
    ('Lokasi tak disebut', 'Kami menyelesaikan kegiatan pada pagi hari.', None),
]
probe_rows = []
for name, text, expected in probes:
    values = reader.features(text)
    school = values['sekolah_utuh_entailment']
    home = values['rumah_utuh_entailment']
    preferred = None if school == home else ('sekolah' if school > home else 'rumah')
    probe_rows.append({'kasus': name, 'teks_sintetis': text, 'arah_diharapkan': expected,
                       'arah_skor': preferred, 'arah_sesuai': None if expected is None else expected == preferred,
                       **values})
display(pd.DataFrame(probe_rows)[['kasus', 'arah_diharapkan', 'arah_skor', 'arah_sesuai',
                                 'sekolah_utuh_entailment', 'rumah_utuh_entailment']].round(4))


## 5 · Baca bukti pada development soal 10

In [ ]:
fold_of = {index: fold['fold'] for fold in manifest['folds'] for index in fold['validation']}
evidence_rows = []
for step, index in enumerate(indices, 1):
    row = bundle.rows[index]
    features = reader.features(row['text'])
    evidence_rows.append({'id_jawaban': row['row_id'], 'baris_excel': row['source_row'],
                          'fold': fold_of[index], 'label_isi': int(bundle.y[index, 0]), **features})
    if step % 20 == 0 or step == len(indices):
        print(f'Inferensi development: {step}/{len(indices)}', flush=True)
evidence = pd.DataFrame(evidence_rows)
score_columns = ['sekolah_utuh_entailment', 'rumah_utuh_entailment']
summary = evidence.groupby('label_isi').agg(n=('id_jawaban', 'size'), **{
    f'median_{column}': (column, 'median') for column in score_columns
}).reset_index()
display(summary.round(4))
print('Review prioritas · semua skor isi 1 dan kasus kerja bakti di rumah dari audit E21.')
display(evidence.loc[evidence['label_isi'].eq(1) | evidence['baris_excel'].eq(331),
                     ['baris_excel', 'label_isi', *score_columns,
                      'sekolah_kutipan_kandidat', 'rumah_kutipan_kandidat']].round(4))


## 6 · Ekspor audit privat

In [ ]:
from tempfile import TemporaryDirectory

path = OUTPUT_ROOT / 'Hasil_AES_Reset_E24_Pilot_Bukti_Konteks.zip'
path.parent.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory() as temp:
    workbook = Path(temp) / '01_Audit_Bukti_Konteks.xlsx'
    metadata = {
        'stage': 'pilot_development', 'model': MODEL_ID, 'model_revision': revision,
        'device': DEVICE, 'torch': version('torch'), 'transformers': version('transformers'),
        'scikit_learn': version('scikit-learn'), 'batch': BATCH, 'max_tokens': MAX_TOKENS,
        'criteria': CRITERIA, 'notebook_code_sha256': NOTEBOOK_CODE_SHA256,
        'split_sha256': manifest['split_sha256'], 'source_sha256': bundle.source_sha256,
        'development_rows': len(evidence), 'test_inference': False, 's3_training': False,
    }
    review = evidence.copy()
    for field in ('lokasi_menurut_teks', 'kutipan_mendukung', 'catatan_review'):
        review[field] = ''
    with pd.ExcelWriter(workbook, engine='openpyxl') as writer:
        summary.to_excel(writer, sheet_name='Ringkasan', index=False)
        pd.DataFrame(probe_rows).to_excel(writer, sheet_name='Contoh Sintetis', index=False)
        review.to_excel(writer, sheet_name='Audit Development Privat', index=False)
        for tab in writer.book:
            tab.freeze_panes = 'B2'
            tab.auto_filter.ref = tab.dimensions
            for row in tab.iter_rows():
                for cell in row:
                    # Simpan kutipan sebagai teks walaupun diawali tanda sama dengan.
                    if cell.data_type == 'f':
                        cell.data_type = 's'
    metadata['workbook_sha256'] = digest(workbook)
    pending = path.with_suffix('.pending')
    with zipfile.ZipFile(pending, 'w', zipfile.ZIP_DEFLATED) as archive:
        archive.write(workbook, workbook.name)
        archive.writestr('manifest.json', json.dumps(metadata, ensure_ascii=False, indent=2))
    pending.replace(path)

print('ZIP audit privat:', path)
if IN_COLAB:
    from google.colab import files
    files.download(str(path))
